# 第 2 章 · 用命令行看懂 turtlesim

本章回答：一个真实的 ROS 系统跑起来是什么样？怎么用 `ros2` 命令行看清楚里面有哪些节点、它们之间怎么通信，还能自己插一脚？

配套讲义：仓库里 `ros2/notes/lessons/02-turtlesim-cli.md`（讲解、动画、练习都在讲义里）。

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。只用 Python 标准库，不需要 ROS 2。

**为什么是模拟**：本机没有 ROS 2。这里只算"消息类型名怎么拆""YAML 里没写的字段是多少""Twist 让乌龟走出什么弧""动作反馈怎么一路变小"这几件事，命令本身要到装好 ROS 2 的 Linux 上敲（里程碑 R-M1）。

In [ ]:
import math

print("准备好了")

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一个函数、跑出来是什么。

| 讲义 | 规则 / 数字 | 写法 | 跑出来 |
|---|---|---|---|
| 第 1 步 类型名 | 包名 / 种类 / 类型名 | `split_type_name(...)` | `turtlesim_msgs` / `srv` / `Spawn` |
| 第 2 步 topic pub | 只写要改的字段，其余是 0 | `fill_defaults(...)` | `linear.y = 0.0` 等 |
| 第 2 步 画弧 | linear.x = 2.0、angular.z = 1.8 | `drive(..., seconds=1.0)` | x ≈ 1.08，y ≈ 1.36，朝向 1.8 rad |
| 第 4 步 弧度 | 1.57 ≈ π/2 = 90° | `math.degrees(-1.57)` | −90.0° |
| 第 4 步 动作 | 反馈 remaining 变小，结果 delta | `rotate_absolute(1.55, -1.57)` | 反馈 −3.12 起；delta 3.12 |

## 1. 先把要用的东西定义好

这几格只是定义（常量、函数、类），运行后没有输出也正常。用到的前面章节的函数也一起抄进来了，不需要本地文件。

**常量 `TWIST_DEFAULT`**

In [ ]:
TWIST_DEFAULT = {"linear": {"x": 0.0, "y": 0.0, "z": 0.0}, "angular": {"x": 0.0, "y": 0.0, "z": 0.0}}

**函数 `drive`**：平面小乌龟：每一小步按朝向前进 linear.x·dt，再转 angular.z·dt。返回 (x, y, theta)。

In [ ]:
def drive(pose: tuple[float, float, float], twist: dict, seconds: float, dt: float = 0.001):
    """平面小乌龟：每一小步按朝向前进 linear.x·dt，再转 angular.z·dt。返回 (x, y, theta)。"""
    x, y, th = pose
    v, w = twist["linear"]["x"], twist["angular"]["z"]
    for _ in range(round(seconds / dt)):
        x += v * math.cos(th) * dt
        y += v * math.sin(th) * dt
        th += w * dt
    return x, y, th

**函数 `fill_defaults`**：`ros2 topic pub` 的 YAML 只写要改的字段，其余取默认值 0。

In [ ]:
def fill_defaults(partial: dict, default: dict) -> dict:
    """`ros2 topic pub` 的 YAML 只写要改的字段，其余取默认值 0。"""
    out = {}
    for key, value in default.items():
        if isinstance(value, dict):
            out[key] = fill_defaults(partial.get(key, {}), value)
        else:
            out[key] = float(partial.get(key, value))
    return out

**函数 `rotate_absolute`**：动作 RotateAbsolute 的模拟：每次转一小步，发反馈 remaining（还差多少），最后返回结果 delta。

In [ ]:
def rotate_absolute(theta_now: float, theta_goal: float, step: float = 0.8):
    """动作 RotateAbsolute 的模拟：每次转一小步，发反馈 remaining（还差多少），最后返回结果 delta。"""
    feedbacks = []
    start = theta_now
    while abs(theta_goal - theta_now) > 1e-9:
        remaining = theta_goal - theta_now
        feedbacks.append(round(remaining, 2))
        theta_now += max(-step, min(step, remaining))  # 一步最多转 step 弧度
    return feedbacks, round(abs(theta_now - start), 2)

**函数 `split_type_name`**：消息类型名三段：包名 / 种类（msg、srv、action）/ 类型名。

In [ ]:
def split_type_name(name: str) -> dict:
    """消息类型名三段：包名 / 种类（msg、srv、action）/ 类型名。"""
    package, kind, type_name = name.split("/")
    assert kind in ("msg", "srv", "action"), kind
    return {"包名": package, "种类": kind, "类型名": type_name}

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
print("\n第一步 · 类型名三段：包名 / 种类 / 类型名")
for name in ["geometry_msgs/msg/Twist", "turtlesim_msgs/srv/Spawn", "turtlesim_msgs/action/RotateAbsolute"]:
    print(" ", name, "→", split_type_name(name))

In [ ]:
print("\n第二步 · topic pub 只写要改的字段，其余是 0")
cmd = fill_defaults({"linear": {"x": 2.0}, "angular": {"z": 1.8}}, TWIST_DEFAULT)
print("  \"{linear: {x: 2.0}, angular: {z: 1.8}}\" →", cmd)

In [ ]:
print("\n第二步（续）· 这条 Twist 让乌龟边走边转，画出一段弧")
x, y, th = drive((0.0, 0.0, 0.0), cmd, seconds=1.0)
print(f"  从原点朝 x 方向出发，走 1 秒：x={x:.3f}, y={y:.3f}, 朝向={th:.3f} rad（= 1.8 × 1 秒）")
r = 2.0 / 1.8
print(f"  解析解：半径 v/ω = {r:.3f}，x = r·sin(1.8) = {r * math.sin(1.8):.3f}，y = r·(1−cos 1.8) = {r * (1 - math.cos(1.8)):.3f}")

In [ ]:
print("\n第三步 · 弧度和角度：1.57 ≈ π/2 = 90°")
print(f"  -1.57 rad = {math.degrees(-1.57):.1f}°；π/2 = {math.pi / 2:.4f}")

In [ ]:
print("\n第四步 · 动作：目标 theta=-1.57，反馈 remaining 一路变小，结果 delta 是实际转过的角度")
feedbacks, delta = rotate_absolute(1.55, -1.57)  # 起始朝向 1.55 是编的，让第一条反馈接近讲义里的 -3.12
print("  反馈:", feedbacks)
print("  结果 delta:", delta)

## 3. 代码里才看得到的

1. **弧的数字**：讲义只说"边走边转、画出一段弧"。代码按 1 ms 一小步积分，和解析解差不到 0.002；turtlesim 自己的积分步长不同，屏幕上的位置会略有差别。
2. **动作的起始朝向 1.55 是编的**，只为让第一条反馈接近讲义输出里的 −3.12；真 turtlesim 每次只转一小点，反馈会多很多条。
3. **`fill_defaults` 是递归的**：Twist 里套着两个 Vector3，所以要一层层往下补 0，这和 `ros2 topic pub` 的 YAML 行为一致。

## 4. 自己改一改

- 把 `angular.z` 改成 0.9，看同样 1 秒弧的半径和终点怎么变；
- 把 `drive` 的 `seconds` 改成 2π/1.8 ≈ 3.49 秒，看乌龟是不是正好回到起点。

在下面的空格子里写，或者直接改上面的格子。讲义末尾的练习先在纸上做，不在这里给答案。

In [ ]:
# 在这里写你的代码

## 口诀

- **套路**：`ros2 <对象> <动作>`；对象是 node / topic / service / param / action / interface。
- **先看后用**：`list -t` 看有什么，`interface show` 看长什么样，再 `pub` / `call` / `set` / `send_goal`。
- **话题三查**：`echo` 看内容，`info` 看谁连着，`hz` 看频率够不够。
- **参数三件套**：`get / set` 现场改，`dump` 存 YAML，`--params-file` 启动时载入。
- **动作三结局**：成功、客户端取消（cancel）、服务端放弃（abort）。